In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, vit_b_16
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR

In [ ]:
# === CONFIGURATION ===
BATCH_SIZE = 32
NUM_EPOCHS = 10
NUM_CLASSES = 8
DATA_DIR = "data_dog_breeds"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === TRANSFORMS ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# === MODEL WRAPPER ===
def get_model(model_name):
    if model_name == "efficientnet":
        model = efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    elif model_name == "vit":
        model = vit_b_16(pretrained=True)
        model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
    else:
        raise ValueError("Unsupported model")
    return model.to(DEVICE)

# === TRAIN FUNCTION ===
def train(model, name):
    print(f"\nTraining {name}...")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = StepLR(optimizer, step_size=3, gamma=0.5)

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss, correct = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

        acc = correct / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Loss: {total_loss:.4f}, Accuracy: {acc:.4f}")
        scheduler.step()

    torch.save(model.state_dict(), f"{name}_dogbreed.pt")

# === MAIN ===
if __name__ == "__main__":
    model1 = get_model("efficientnet")
    train(model1, "efficientnet")

    model2 = get_model("vit")
    train(model2, "vit")
